In [20]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from pathlib import Path
import io
import requests


In [26]:
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
service = Service(ChromeDriverManager().install())
browser = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(browser, 20)

In [27]:
excel_path = Path("data/challenge.xlsx")
if not excel_path.exists():
    download_url = "https://rpachallenge.com/assets/downloadFiles/challenge.xlsx"
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://rpachallenge.com/"
    }
    response = requests.get(download_url, headers=headers, timeout=30)
    response.raise_for_status()
    df = pd.read_excel(io.BytesIO(response.content))
    excel_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(excel_path, index=False)
else:
    df = pd.read_excel(excel_path)
df.rename(columns = {'Last Name ':'Last Name'}, inplace=True)

In [28]:
website = "https://rpachallenge.com"
browser.get(website)
start_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button")))
start_button.click()
for i in range(df.shape[0]):
    print('page: ', i)
    row = df.iloc[i,:].to_dict()
    labels = browser.find_elements(By.CSS_SELECTOR, 'label')
    for label in labels:
        field_name = label.text.strip()
        value = str(row[field_name])
        print(f'  {field_name}: {value}')
        xpath = f"//label[normalize-space()='{field_name}']/following-sibling::input"
        inputfield = browser.find_element(By.XPATH, xpath)
        inputfield.clear()
        inputfield.send_keys(value)

    submitbtn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR,'input[type="submit"]')))
    submitbtn.click()

page:  0
  Email: jsmith@itsolutions.co.uk
  Role in Company: Analyst
  Phone Number: 40716543298
  Company Name: IT Solutions
  Address: 98 North Road
  First Name: John
  Last Name: Smith
page:  1
  Company Name: MediCare
  Phone Number: 40791345621
  Address: 11 Crown Street
  Role in Company: Medical Engineer
  First Name: Jane
  Email: jdorsey@mc.com
  Last Name: Dorsey
page:  2
  Last Name: Kipling
  Company Name: Waterfront
  Email: kipling@waterfront.com
  First Name: Albert
  Address: 22 Guild Street
  Phone Number: 40735416854
  Role in Company: Accountant
page:  3
  Email: mrobertson@mc.com
  Address: 17 Farburn Terrace
  Role in Company: IT Specialist
  Company Name: MediCare
  Phone Number: 40733652145
  First Name: Michael
  Last Name: Robertson
page:  4
  Phone Number: 40799885412
  Email: dderrick@timepath.co.uk
  Role in Company: Analyst
  First Name: Doug
  Company Name: Timepath Inc.
  Address: 99 Shire Oak Road
  Last Name: Derrick
page:  5
  Last Name: Marlowe
  Co

In [24]:
browser.quit()